## self-sustained activity in a recurrent neural network using Brian2

Here's a Python implementation of a recurrent network of 100 LIF neurons with random delays and sustained activity using Brian2:


In [ ]:
import numpy as np
import brian2 as sim
import matplotlib.pyplot as plt

# Set up the simulation
sim.defaultclock.dt = 0.1*sim.ms
sim.start_scope()

# Network parameters
N = 100  # Number of neurons
duration = 500*sim.ms  # Simulation duration

# LIF neuron parameters
tau_m = 20*sim.ms      # Membrane time constant
V_rest = -70*sim.mV    # Resting potential
V_th = -50*sim.mV      # Spike threshold
V_reset = -65*sim.mV   # Reset potential
R_m = 100*sim.Mohm     # Membrane resistance
tau_refractory = 2*sim.ms  # Refractory period

# Synaptic parameters
delay_min = 2*sim.ms   # Minimum synaptic delay
delay_max = 40*sim.ms  # Maximum synaptic delay
synaptic_weight = 0.5*sim.mV  # Base synaptic weight
sparsity = 0.2     # Connection probability (20% connectivity)

# Define LIF neuron equations
eqs = '''
dv/dt = (-(v - V_rest) + R_m*I_syn)/tau_m : volt (unless refractory)
I_syn : amp  # Synaptic current input
'''

# Create neuron group
neurons = sim.NeuronGroup(N, eqs,
                          threshold='v > V_th',
                          reset='v = V_reset',
                          refractory=tau_refractory,
                          method='euler')

# Initialize membrane potentials randomly between resting and threshold
neurons.v = V_rest + (V_th - V_rest) * np.random.rand(N)

# Create synapses with random delays
synapses = sim.Synapses(neurons, neurons,
                        'w : volt',  # Synaptic weight variable
                        on_pre='v_post += w')  # When pre-synaptic neuron fires

# Connect neurons sparsely (20% probability)
synapses.connect(p=sparsity)

# Assign random synaptic weights and delays
synapses.w = synaptic_weight * np.random.exponential(1, size=len(synapses))
synapses.delay = delay_min + (delay_max - delay_min) * np.random.rand(len(synapses))

# Add external Poisson input to initiate activity
poisson_rate = 50*sim.Hz  # Rate of external inputs
external_input = sim.PoissonInput(neurons, 'v', N=20, rate=poisson_rate, weight=1*sim.mV)

# Monitor spikes for visualization
spike_monitor = sim.SpikeMonitor(neurons)
state_monitor = sim.StateMonitor(neurons, 'v', record=[0])  # Record first neuron

# Run simulation in two phases: stimulation then free-running
stim_duration = 50*sim.ms
sim.run(stim_duration)  # Initial stimulation

# Remove external input for sustained activity observation
del external_input
sim.run(duration - stim_duration)  # Continue without external input

# Plot results
plt.figure(figsize=(12, 8))

# Raster plot of spikes
plt.subplot(3, 1, 1)
plt.plot(spike_monitor.t/sim.ms, spike_monitor.i, '.k', markersize=1)
plt.xlabel('Time (ms)')
plt.ylabel('Neuron index')
plt.title('Spike Raster Plot')

# Firing rate over time (binned)
plt.subplot(3, 1, 2)
bin_size = 10*sim.ms
bins = np.arange(0, duration/sim.ms, bin_size/sim.ms)
spike_times_ms = spike_monitor.t/sim.ms
histogram, _ = np.histogram(spike_times_ms, bins=bins)
plt.plot(bins[:-1], histogram/bin_size/N*1000, 'b-', linewidth=1)
plt.xlabel('Time (ms)')
plt.ylabel('Population firing rate (Hz)')
plt.title('Population Firing Rate')

# Sample membrane potential trace
plt.subplot(3, 1, 3)
plt.plot(state_monitor.t/sim.ms, state_monitor.v[0]/sim.mV, 'r-', linewidth=0.8)
plt.xlabel('Time (ms)')
plt.ylabel('Membrane potential (mV)')
plt.title('Sample Neuron Membrane Potential')

plt.tight_layout()
plt.show()

# Print network statistics
print(f"Total spikes recorded: {len(spike_monitor.t)}")
print(f"Mean firing rate: {len(spike_monitor.t)/(N*(duration/sim.second)):.2f} Hz/neuron")
print(f"Connection density: {len(synapses)/N**2*100:.1f}%")

## using pyNN

In [ ]:
import numpy as np
import pyNN.brian as sim
import matplotlib.pyplot as plt

# Set up the simulation
sim.setup(timestep=0.1, min_delay=2.0, max_delay=40.0)

# Network parameters
N = 100  # Number of neurons
duration = 500.0  # Simulation duration in ms

# LIF neuron parameters
neuron_params = {
    'cm': 0.25,        # nF - membrane capacitance
    'i_offset': 0.0,   # nA - offset current
    'tau_m': 20.0,     # ms - membrane time constant
    'tau_refrac': 2.0, # ms - refractory period
    'tau_syn_E': 5.0,  # ms - excitatory synaptic time constant
    'tau_syn_I': 5.0,  # ms - inhibitory synaptic time constant
    'v_reset': -65.0,  # mV - reset potential
    'v_rest': -70.0,   # mV - resting potential
    'v_thresh': -50.0  # mV - spike threshold
}

# Create neuron population
neurons = sim.Population(N, sim.IF_curr_exp(**neuron_params))

# Initialize membrane potentials randomly
neurons.initialize(v=rand_dist)

# Create random membrane potential distribution
rand_dist = sim.RandomDistribution('uniform', [-70.0, -50.0])
neurons.initialize(v=rand_dist)

# Create recurrent connections with random delays
connector = sim.FixedProbabilityConnector(0.2)  # 20% connectivity

# Synaptic parameters
synapse_params = {
    'weight': 0.5,  # nA - synaptic weight
    'delay': sim.RandomDistribution('uniform', [2.0, 40.0])  # ms - random delays
}

# Create projection
projection = sim.Projection(neurons, neurons, connector, 
                           sim.StaticSynapse(**synapse_params),
                           receptor_type='excitatory')

# Add external Poisson input to initiate activity
poisson_params = {
    'rate': 50.0,  # Hz - Poisson rate
    'start': 0.0,  # ms - start time
    'duration': 50.0  # ms - duration of input
}

poisson_input = sim.Population(20, sim.SpikeSourcePoisson(**poisson_params))

# Connect Poisson input to network
input_projection = sim.Projection(poisson_input, neurons, 
                                 sim.OneToOneConnector(),
                                 sim.StaticSynapse(weight=1.0, delay=1.0))

# Record spikes and membrane potentials
neurons.record(['spikes', 'v'])
# Also record from a subset for detailed analysis
sample_neurons = neurons[0:1]  # Record from first neuron

# Run simulation
sim.run(duration)

# Extract data
spikes = neurons.get_data('spikes')
voltage = neurons.get_data('v')

# Plot results
plt.figure(figsize=(12, 8))

# Raster plot of spikes
plt.subplot(3, 1, 1)
for segment in spikes.segments:
    for i, spiketrain in enumerate(segment.spiketrains):
        plt.plot(spiketrain.times, [i]*len(spiketrain), '.k', markersize=1)
plt.xlabel('Time (ms)')
plt.ylabel('Neuron index')
plt.title('Spike Raster Plot')

# Firing rate over time (binned)
plt.subplot(3, 1, 2)
all_spike_times = []
for segment in spikes.segments:
    for spiketrain in segment.spiketrains:
        all_spike_times.extend(spiketrain.times)

if len(all_spike_times) > 0:
    bin_size = 10.0
    bins = np.arange(0, duration, bin_size)
    histogram, _ = np.histogram(all_spike_times, bins=bins)
    firing_rate = histogram / bin_size * 1000 / N  # Convert to Hz/neuron
    plt.plot(bins[:-1], firing_rate, 'b-', linewidth=1)
plt.xlabel('Time (ms)')
plt.ylabel('Population firing rate (Hz)')
plt.title('Population Firing Rate')

# Sample membrane potential trace
plt.subplot(3, 1, 3)
for segment in voltage.segments:
    if len(segment.analogsignals) > 0:
        v_signal = segment.analogsignals[0]
        time_array = v_signal.times
        v_array = v_signal[:, 0]  # First neuron
        plt.plot(time_array, v_array, 'r-', linewidth=0.8)
        break
plt.xlabel('Time (ms)')
plt.ylabel('Membrane potential (mV)')
plt.title('Sample Neuron Membrane Potential')

plt.tight_layout()
plt.show()

# Print network statistics
total_spikes = sum(len(segment.spiketrains[0]) for segment in spikes.segments 
                  for segment in [spikes.segments[0]] if spikes.segments)
print(f"Total spikes recorded: {sum(len(spiketrain) for segment in spikes.segments for spiketrain in segment.spiketrains)}")
print(f"Mean firing rate: {sum(len(spiketrain) for segment in spikes.segments for spiketrain in segment.spiketrains) / (N * duration/1000):.2f} Hz/neuron")
print(f"Number of connections: {projection.size()}")
print(f"Connection density: {projection.size()/(N*N)*100:.1f}%")

# Clean up
sim.end()

In [ ]:
%pip install pynn brian2


